In [1]:
import os
from dotenv import load_dotenv


##=============== IMPORT TO LANGSMITH ======================
load_dotenv("apikey.env")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")

In [2]:
from langchain_deepseek import ChatDeepSeek
from langchain.storage import LocalFileStore
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

In [3]:
#=============== 设置模型的对应的 embeddings =================#

BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
# LANGSMITH_API_KEY = os.getenv("LANGSMITH-API-KEY")

deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")

model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

SECESSFULLY!


In [4]:
#=============== 载入数据，并且选择合适的方法拆分成快===============
# !!!!!!! load() 适合小数据jupyter加载使用，
# 数据量过大请使用 lazy_load() / alazy_load()

raw_documents = DirectoryLoader('./state2/example/corpus/', 
                                glob="**/*.txt",
                                show_progress=True,
                                use_multithreading=True,
                                loader_cls=TextLoader,
                                loader_kwargs={
                                    "encoding": "utf-8",
                                }
                                )
load_doc = raw_documents.lazy_load()


In [5]:
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
documents = text_splitter.split_documents(
                                    list(load_doc)
                                    )

100%|███████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 109.45it/s]
Created a chunk of size 2622, which is longer than the specified 2000
Created a chunk of size 3339, which is longer than the specified 2000
Created a chunk of size 2941, which is longer than the specified 2000
Created a chunk of size 4052, which is longer than the specified 2000
Created a chunk of size 2240, which is longer than the specified 2000
Created a chunk of size 3435, which is longer than the specified 2000
Created a chunk of size 3142, which is longer than the specified 2000
Created a chunk of size 3035, which is longer than the specified 2000
Created a chunk of size 4081, which is longer than the specified 2000
Created a chunk of size 4316, which is longer than the specified 2000
Created a chunk of size 3353, which is longer than the specified 2000
Created a chunk of size 2961, which is longer than the specified 2000
Created a chunk of size 3201, which is l

In [6]:
len(documents)

1746

In [8]:
#=================== 建立向量数据库和本地缓存 ======================#

underlying_embeddings = HuggingFaceBgeEmbeddings(model_name="moka-ai/m3e-base")
store = LocalFileStore("./cache/FourGreatClasissDB")
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings, 
    store, 
    namespace=underlying_embeddings.model_name
)
simpleDB = FAISS.from_documents(documents, cached_embedder)

C:\Users\hhm18\miniconda3\envs\TrainingCamp\lib\site-packages\langchain\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [9]:
selected_data = list(store.yield_keys())[:2]
print(f"缓存记录: {len(list(store.yield_keys()))}, \n\n  展示前几条：\n{selected_data}")

缓存记录: 1746, 

  展示前几条：
['moka-ai\\m3e-base001fe514-5002-54ad-8bf2-a2f60ca5953f', 'moka-ai\\m3e-base00bc9fc1-9f4c-56b4-a8d9-b575385c3922']


这里考虑到对于提问者并不完全了解数据库里面的内容，如果提出的总结性的问题我们应该如何解答？
- 问题粗略分解为完全根据事实回答的

- 以及总结推断类型的问题

    - 考虑使用三种不同类型的prompt
    - 1. classifier： 将问题分为 insight and rag
    - 2. 使用 route 进行传递
    - 3. 最后生成给用户的答案

In [10]:
# 提示模板
classify_template = """
你是一个资深的文学类别专家，擅长文学的思考。请你根据用户给出的问题判断，
这个问题属于是以下的哪一类型（label）：
- fact：问题询问具体事实、数据、定义，需严格依据文档回答。
- insight； 问题要求总结、推断、解释意义、预测趋势、表达观点等，允许结合常识推理。

问题：{question}
类别（只能输出 fact 或 insight）
"""
classify_prompt = ChatPromptTemplate.from_template(classify_template)

classify_chain = classify_prompt | model | StrOutputParser()


In [11]:
# 检索器
insight_retriever = simpleDB.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)  # 检索最相关的3个chunks


# 提示模板
insight_template = """
你是一位资深分析师。请结合以下上下文，并运用你的常识和推理能力，回答问题。
可以总结、推断、解释深层含义，但请明确区分哪些来自资料，哪些是你的分析。
并且你总是以下内容作为开头：按照我的理解

上下文：
{context}

问题：{question}
"""
insight_prompt = ChatPromptTemplate.from_template(insight_template)

# 构建 RAG 链
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

insight_chain = (
    {"context": RunnableLambda(lambda x: x["question"]) | insight_retriever | format_docs, 
     "question": RunnablePassthrough()}
    | insight_prompt
    | model
    | StrOutputParser()
)

In [12]:
# 检索器
rag_retriever = simpleDB.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)  # 检索最相关的3个chunks


# 提示模板
rag_template = """
你是一个专业、准确的问答助手。请仅基于以下提供的上下文回答问题。
如果上下文不足以回答，请说“根据现有资料无法回答”，并且给出为什么不足以回答的原因。

上下文：
{context}

问题：{question}
"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# 构建 RAG 链
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": RunnableLambda(lambda x: x["question"]) | rag_retriever | format_docs, 
     "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

In [13]:
from langchain_core.runnables import RunnableBranch

def is_insight(input_dict):
    question = input_dict["question"]
    print("🔍 输入到分类器:", question)
    result = classify_chain.invoke({"question": question})
    print("🎯 分类结果:", repr(result))
    return "insight" in result.strip().lower()

full_chain = RunnableBranch(
    (is_insight, insight_chain),
    rag_chain
)

In [14]:
from langchain_community.callbacks import get_openai_callback

In [15]:
with get_openai_callback() as cb1:
    query = {"question": "孙行者是谁？"}

    async for chunk in rag_chain.astream(query):
        print(chunk, end="", flush=True)
    print(f"\n{cb1}\n")
    print("\n==============================================\n")

with get_openai_callback() as cb2:
    async for chunk in full_chain.astream(query):
        print(chunk, end="", flush=True)
    print(f"\n{cb2}\n")

根据现有资料无法回答。

**原因**：  
提供的上下文中虽然多次出现“孙行者”这一称呼，但并未提供任何关于其身份背景的直接介绍。上下文主要描述了孙行者在取经路上的种种经历和对话，但缺乏对他出身、来历等基本身份信息的说明。因此，仅凭给定资料无法回答“孙行者是谁”这一问题。
Tokens Used: 11292
	Prompt Tokens: 11216
		Prompt Tokens Cached: 0
	Completion Tokens: 76
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0



🔍 输入到分类器: 孙行者是谁？
🎯 分类结果: 'fact'
根据现有资料无法回答。

原因：提供的上下文中虽然多次出现“孙行者”这一称呼，但并未包含任何直接说明“孙行者”真实身份或背景来历的客观描述性语句。上下文主要是《西游记》的故事情节片段，其中“孙行者”是作为一个既定角色在对话和叙述中被使用的，但缺乏关于“他是谁”的定义性信息。
Tokens Used: 11390
	Prompt Tokens: 11309
		Prompt Tokens Cached: 11264
	Completion Tokens: 81
		Reasoning Tokens: 0
Successful Requests: 2
Total Cost (USD): $0.0



In [17]:
with get_openai_callback() as cb1:
    query = {"question": "请你推断唐僧师徒一共经历了多少磨难？"}

    async for chunk in rag_chain.astream(query):
        print(chunk, end="", flush=True)
    print(f"\n{cb1}\n")
    print("\n==============================================\n")

with get_openai_callback() as cb2:
    async for chunk in full_chain.astream(query):
        print(chunk, end="", flush=True)
    print(f"\n{cb2}\n")

根据现有资料无法回答。

原因：虽然上下文第九十九回中列出了“圣僧历难簿”，并明确记载了从“金蝉遭贬第一难”到“凌云渡脱胎八十难”的具体名目，但随后观音菩萨指出“佛门中九九归真，圣僧受过八十难，还少一难，不得完成此数”，并因此令揭谛追加了一难。然而，上下文并未明确说明这最后一难的具体名称或内容，也未给出最终磨难的总数。因此，仅凭提供的资料无法推断出唐僧师徒经历的准确磨难总数。
Tokens Used: 11214
	Prompt Tokens: 11091
		Prompt Tokens Cached: 1280
	Completion Tokens: 123
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0



🔍 输入到分类器: 请你推断唐僧师徒一共经历了多少磨难？
🎯 分类结果: 'insight'
按照我的理解，根据提供的上下文，特别是第九十九回中菩萨查阅的"灾难簿子"明确记载，唐僧师徒在取经路上共经历了八十难。以下是详细分析：

**来自资料的依据**：  
1. 第九十九回中，护教伽蓝向菩萨呈报的灾难簿子清晰列出了从"金蝉遭贬第一难"到"凌云渡脱胎八十难"的具体名称，并总结为"圣僧历难簿分明"。  
2. 菩萨核查后指出："圣僧受过八十难，还少一难，不得完成此数"，随后通过金刚施法补足最后一难（通天河落水），最终凑齐八十一难。

**我的分析和推理**：  
- 虽然灾难簿子仅列至第八十难，但结合上下文，菩萨为符合"九九归真"的佛教理念，特意追加一难，使总数达到八十一难。  
- 最终磨难数（八十一）具有象征意义：在佛教文化中，"九九"代表圆满，呼应文中"九九数完魔灭尽"的表述，体现修行需经历完整劫数方能成道。  
- 需注意，部分章回（如第二十回、第五十一回）描述的个别劫难可能已包含在灾难簿子的八十难之内，而非独立新增。

**结论**：  
唐僧师徒经历的磨难总数应为**八十一难**。这一数字既源于灾难簿子的明确记载，也通过菩萨的干预达成宗教意义上的圆满。
Tokens Used: 14461
	Prompt Tokens: 14141
		Prompt Tokens Cached: 1344
	Completion Tokens: 320
		Reasoning

In [ ]:
## 回溯原文

text1 = '''按照我的理解，这个问题需要结合《西游记》原著中的情节和主题进行分析。以下是我的回答，分为"来自资料的依据"和"我的分析"两部分：

---

### 来自资料的依据
1. **修行历程**：  
   - 第一回记载孙悟空本是花果山仙石所化的石猴，后拜师菩提祖师修得长生之道、七十二变和筋斗云，奠定修行基础。  
   - 多次提及他保护唐僧西天取经，如第二十回、第五十七回等，虽屡遭误解（如被唐僧驱逐），但仍坚持护师。  

2. **功绩与磨难**：  
   - 第九十九回明确列出唐僧师徒经历的“八十一难”，孙悟空在降妖除魔中贡献关键力量（如斗黄风怪、平顶山逢魔等）。  
   - 最终回提到，如来佛祖因取经团队“功行圆满”，封孙悟空为“斗战胜佛”。  

3. **心性转变**：  
   - 第五十七回中，孙悟空被唐僧驱逐后向菩萨诉苦，仍选择回归团队，体现对使命的忠诚。  
   - 第二十回等章节中，他逐渐收敛桀骜，学会忍耐（如忍受紧箍咒之苦）。  

---

### 我的分析
1. **修行与悟性的结合**：  
   孙悟空的成佛并非偶然。他天生灵根（仙石孕育），又主动求道（拜师菩提），具备超凡的悟性和执行力。这种“先天资质+后天努力”的模式，符合佛教“众生皆可成佛”的思想，但需主动修行。  

2. **磨难中的成长**：  
   取经路上的磨难（如师徒矛盾、妖魔阻挠）实质是心性考验。孙悟空从最初的 impulsive（如随意打死强盗）到后期懂得权衡（如请神仙相助而非一味硬斗），体现了佛教“降伏其心”的修行核心。他的战斗力固然重要，但真正关键是他在过程中逐渐破除“我执”。  

3. **团队角色与因果逻辑**：  
   孙悟空在团队中承担“解决问题者”的角色，而如来封佛时强调“功行圆满”。这暗示成佛需两大条件：  
   - **外在功绩**：护唐僧取经，普度众生。  
   - **内在觉悟**：通过磨难磨去戾气，如他对唐僧的忠诚、对使命的坚持，最终达到“无挂碍”境界（呼应《心经》主题）。  

4. **作者的价值取向**：  
   吴承恩通过孙悟空成佛的结局，传递了“放下屠刀，立地成佛”的佛教理念，但更强调“修行需历劫”。孙悟空的逆袭（从妖到佛）打破了出身论，彰显了明代心学“人皆可为圣贤”的思想影响。  

---

### 结论
孙悟空能成佛，既因他护经的客观功绩被如来认可（资料依据），也因他在十四年取经路上完成了心性升华，从“妄心”走向“真心”（分析推论）。这一过程融合了道教的修炼基础与佛教的顿悟理念，成为《西游记》“心性修持”主题的具象化体现。'''

docs_text = simpleDB.similarity_search(text1, k=10) # 使用和insight相同的k

In [ ]:
print(format_docs(docs_text[-3:]))

In [ ]:
# query = {"question": "诸葛亮是谁？在三国演义中有什么关于他的重要事件？"}

# async for chunk in rag_chain.astream(query):
#     print(chunk, end="", flush=True)

# print("\n==============================================\n")

# async for chunk in full_chain.astream(query):
#     print(chunk, end="", flush=True)